In [ ]:
import os
import shutil
import random
from pathlib import Path

SOURCE_DIR = "augmented_poker_data"
OUTPUT_SUITS = "dataset_suits"
OUTPUT_RANKS = "dataset_ranks"
SPLIT_RATIO = 0.8

def prepare_data():
    for root_dir in [OUTPUT_SUITS, OUTPUT_RANKS]:
        for split in ['train', 'val']:
            os.makedirs(os.path.join(root_dir, split), exist_ok=True)

    suits = [d for d in os.listdir(SOURCE_DIR) if os.path.isdir(os.path.join(SOURCE_DIR, d))]

    for suit in suits:
        suit_path = os.path.join(SOURCE_DIR, suit)
        ranks = [d for d in os.listdir(suit_path) if os.path.isdir(os.path.join(suit_path, d))]
        
        for rank in ranks:
            rank_path = os.path.join(suit_path, rank)
            images = [f for f in os.listdir(rank_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
            
            random.shuffle(images)
            split_idx = int(len(images) * SPLIT_RATIO)
            train_imgs = images[:split_idx]
            val_imgs = images[split_idx:]
            
            def copy_files(file_list, split_type):
                for img in file_list:
                    src_file = os.path.join(rank_path, img)
                    
                    dest_suit_dir = os.path.join(OUTPUT_SUITS, split_type, suit)
                    os.makedirs(dest_suit_dir, exist_ok=True)
                    shutil.copy(src_file, os.path.join(dest_suit_dir, f"{rank}_{img}"))
                    
                    dest_rank_dir = os.path.join(OUTPUT_RANKS, split_type, rank)
                    os.makedirs(dest_rank_dir, exist_ok=True)
                    shutil.copy(src_file, os.path.join(dest_rank_dir, f"{suit}_{img}"))

            copy_files(train_imgs, 'train')
            copy_files(val_imgs, 'val')

    print("Data preparation complete!")
    print(f"Suits dataset at: {OUTPUT_SUITS}")
    print(f"Ranks dataset at: {OUTPUT_RANKS}")


prepare_data()

Data preparation complete!
Suits dataset at: dataset_suits
Ranks dataset at: dataset_ranks


In [ ]:
import torch
from ultralytics import YOLO


def train_suit_model():
    device = 0 if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {torch.cuda.get_device_name(0) if device == 0 else 'CPU'}")

    model = YOLO('yolov8n-cls.pt')

    results = model.train(
        data='dataset_suits',
        epochs=50,
        imgsz=224,
        batch=32,
        project='poker_project',
        name='suit_model',
        device=device,
        workers=6,
        pretrained=True
    )

    metrics = model.val(device=device)
    print(f"Top-1 Accuracy: {metrics.top1}")


train_suit_model()

Using device: NVIDIA GeForce RTX 4060 Laptop GPU
New https://pypi.org/project/ultralytics/8.4.14 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.202  Python-3.12.1 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset_suits, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-cls.pt, momentum=0.937, mosaic=1.0, mult

In [ ]:
from ultralytics import YOLO
import torch

def train_rank_model():
    model = YOLO('yolov8n-cls.pt') 

    device = 0 if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {torch.cuda.get_device_name(0) if device == 0 else 'CPU'}")
    
    results = model.train(
        data='dataset_ranks',
        epochs=50,
        imgsz=224,
        batch=8,
        workers=4,
        project='poker_project',
        name='rank_model',
        device=device,
    )

    metrics = model.val(device=device)
    print(f"Top-1 Accuracy: {metrics.top1}")


train_rank_model()

Using device: NVIDIA GeForce RTX 4060 Laptop GPU
New https://pypi.org/project/ultralytics/8.4.14 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.202  Python-3.12.1 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset_ranks, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-cls.pt, momentum=0.937, mosaic=1.0, multi